In [4]:
# --- 必要なパッケージ ---
library(caret)
library(pls)
library(Metrics)
library(ggplot2)
library(dplyr)

# --- VIP計算関数 (変更なし) ---
calc_vip <- function(pls_model, ncomp) {
  max_comp <- min(ncomp, nrow(pls_model$Yloadings))
  SS <- pls_model$Yloadings[1:max_comp, , drop = FALSE]^2
  W <- pls_model$loading.weights[, 1:max_comp, drop = FALSE]
  p <- nrow(W)
  Wnorm2 <- colSums(W^2)
  SSY <- sum(SS)
  
  vip_scores <- numeric(p)
  for (j in 1:p) {
    weight <- 0
    for (a in 1:max_comp) {
      if (Wnorm2[a] == 0) next
      weight <- weight + SS[a] * (W[j, a]^2 / Wnorm2[a])
    }
    vip_scores[j] <- sqrt(p * weight / SSY)
  }
  names(vip_scores) <- rownames(W)
  return(vip_scores)
}

# --- OOD分割関数 (変更なし) ---
create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
  df_features <- df[, feature_cols, drop = FALSE]
  dist_matrix <- dist(df_features, method = "euclidean")
  avg_distances <- as.matrix(dist_matrix) %>% colMeans()
  num_ood <- floor(nrow(df) * ood_ratio)
  if (num_ood == 0 && nrow(df) > 0) num_ood <- 1
  ood_indices <- order(avg_distances, decreasing = TRUE)[1:num_ood]
  test_set <- df[ood_indices, ]
  train_set <- df[-ood_indices, ]
  cat(paste0("  OOD Split: Total=", nrow(df), ", Train=", nrow(train_set), ", Test(OOD)=", nrow(test_set), "\n"))
  return(list(train = train_set, test = test_set, ood_indices = ood_indices))
}

# --- 設定 ---
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

# 【重要】除外する列のリスト
excluded_cols <- c("X") 

base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_ncomp", "R2", "RMSE", "MAE", "RPD", "Q2", "n_samples", "n_features", "OOD_R2", "OOD_RMSE")

result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# --- メインループ ---
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, " (Excluding ID: X) ===\n")
  
  # 1. データの読み込み
  df_all <- read.csv(filepath)
  
  # 【重要修正】ID列（X）が存在すれば即座に削除
  df_all <- df_all[, setdiff(colnames(df_all), excluded_cols), drop = FALSE]
  
  feature_vars_initial <- setdiff(colnames(df_all), target_vars)
  
  for (target_var in target_vars) {
    cat("\n--- Training PLS for:", target_var, " ---\n")
    df_temp <- df_all[, c(feature_vars_initial, target_var)]
    df_temp <- df_temp[complete.cases(df_temp), ]
    
    # 処理後の特徴量リストを更新
    feature_vars_final <- setdiff(colnames(df_temp), target_var)
    
    if (nrow(df_temp) < 20) {
      cat("  Skipping: insufficient data.\n")
      next
    }
    
    # === 2. OOD分割 (Xを除いた特徴量で実施) ===
    split_list <- create_ood_split(df_temp, feature_vars_final, ood_ratio = 0.1) 
    df_train_cv <- split_list$train
    df_ood_test <- split_list$test
    
    # === 3. PLSモデルの訓練 ===
    max_ncomp <- min(10, ncol(df_train_cv) - 1, nrow(df_train_cv) - 1)
    if (max_ncomp < 1) next
    
    tune_grid <- expand.grid(ncomp = 1:max_ncomp)
    ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final") 
    
    model <- train(
      formula(paste(target_var, "~ .")),
      data = df_train_cv,
      method = "pls",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid,
      preProcess = c("center", "scale")
    )
    
    # === 4. 結果の計算と格納 ===
    cv_preds <- model$pred
    best_params <- model$bestTune
    for (param in names(best_params)) {
      cv_preds <- cv_preds[cv_preds[[param]] == best_params[[param]], ]
    }
    
    if (nrow(cv_preds) > 0) {
      pred <- cv_preds$pred
      obs <- cv_preds$obs
      R2 <- round(cor(obs, pred)^2, 3)
      RMSE_val <- round(rmse(obs, pred), 3)
      MAE_val <- round(mae(obs, pred), 3)
      RPD_val <- round(sd(obs) / RMSE_val, 3)
      
      press <- sum((obs - pred)^2)
      tss <- sum((obs - mean(obs))^2)
      Q2_val <- round(1 - press / tss, 3)
    } else {
      R2 <- RMSE_val <- MAE_val <- RPD_val <- Q2_val <- NA
    }
    
    best_ncomp <- model$bestTune$ncomp
    result_matrix[paste0("Best_ncomp_", target_var), file] <- best_ncomp
    result_matrix[paste0("R2_", target_var), file] <- R2
    result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file] <- MAE_val
    result_matrix[paste0("RPD_", target_var), file] <- RPD_val
    result_matrix[paste0("Q2_", target_var), file] <- Q2_val
    result_matrix[paste0("n_samples_", target_var), file] <- nrow(df_temp)
    result_matrix[paste0("n_features_", target_var), file] <- length(feature_vars_final)
    
    # OOD性能
    if (!is.null(df_ood_test) && nrow(df_ood_test) > 0) {
      ood_preds <- predict(model, newdata = df_ood_test)
      ood_obs <- df_ood_test[[target_var]]
      ood_R2 <- round(cor(ood_obs, ood_preds)^2, 3)
      ood_RMSE_val <- round(rmse(ood_obs, ood_preds), 3)
      result_matrix[paste0("OOD_R2_", target_var), file] <- ood_R2
      result_matrix[paste0("OOD_RMSE_", target_var), file] <- ood_RMSE_val
    }
    
    # === 5. 保存 (fixed_ を付与) ===
    saveRDS(list(model = model, training_data = df_train_cv, ood_test_data = df_ood_test), 
            file = paste0("fixed20250625_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".rds"))
    
    # VIPスコアの保存
    vip <- calc_vip(model$finalModel, best_ncomp)
    write.csv(data.frame(Variable = names(vip), VIP = vip), 
              paste0("fixed20250625_vip_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".csv"), row.names = FALSE)
  }
}

# --- 最終結果の保存 ---
output_file <- paste0("fixed20250625_pls_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\n✅ Fixed PLS Summary saved as:", output_file, "\n")


=== Processing: n_base.csv  (Excluding ID: X) ===

--- Training PLS for: Jsc  ---
  OOD Split: Total=358, Train=323, Test(OOD)=35

--- Training PLS for: Voc  ---
  OOD Split: Total=358, Train=323, Test(OOD)=35

--- Training PLS for: FF  ---
  OOD Split: Total=358, Train=323, Test(OOD)=35

--- Training PLS for: PCEmax  ---
  OOD Split: Total=358, Train=323, Test(OOD)=35

=== Processing: n_base_OH.csv  (Excluding ID: X) ===

--- Training PLS for: Jsc  ---
  OOD Split: Total=358, Train=323, Test(OOD)=35


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICEM, SMILESsnamenM_ICMA, SMILESsnamenM_ICMM, SMILESsnamenM_MOPFP, SMILESsnamenM_NCMA, SMILESsnamenM_NCMM, SMILESsnamenM_PCBM, SMILESsnamenM_PFP, SMILESsnamenM_TC61BM, SMILESsnamenM_TC61PM, SMILESsnamenM_TFP, SMILESsnamep1M_BDTH1, SMILESsnamep1M_BDTH11, SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH6, SMILESsnamep1M_BDTH7, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_DTSCh, SMILESsnamep1M_DTTPTP1, SMILESsnamep1M_PBDTTPD, SMILESsnamep1M_PBDTTS2, SMILESsnamep1M_PCDTBX, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PCDTPT, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PCDTQx, SMILESsnamep1M_PCDTTPP, SMILESsnamep1M_PDTPDTDPP, SMILESsnamep1M_PFBTBDT, SMILESsnamep1M_PFDTBT, SMILESsnamep1M_PTB1, SMILESsnamep1M_PTB2, SMILESsnamep1M_PTB6, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2.thienyl..1.2.5.oxadiazolo.3.4d.pyridazine.P1, SMILESsnam

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICEM, SMILESsnamenM_ICMA, SMILESsnamenM_ICMM, SMILESsnamenM_MOPFP, SMILESsnamenM_NCMA, SMILESsnamenM_NCMM, SMILESsnamenM_PCBM, SMILESsnamenM_PFP, SMILESsnamenM_TC61BM, SMILESsnamenM_TC61PM, SMILESsnamenM_TFP, SMILESsnamep1M_BDTH1, SMILESsnamep1M_BDTH11, SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH6, SMILESsnamep1M_BDTH7, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_DTSC6, SMILESsnamep1M_DTSCh, SMILESsnamep1M_DTTPTP1, SMILESsnamep1M_PBDTTPD, SMILESsnamep1M_PCDTBX, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PCDTPT, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PCDTQx, SMILESsnamep1M_PCDTTPP, SMILESsnamep1M_PDTPDTDPP, SMILESsnamep1M_PFDTBT, SMILESsnamep1M_PTB1, SMILESsnamep1M_PTB2, SMILESsnamep1M_PTB6, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2.thienyl..1.2.5.oxadiazolo.3.4d.pyridazine.P1, SMILESsnamep1M_Poly.N9.fheptadecanyl


--- Training PLS for: Voc  ---
  OOD Split: Total=358, Train=323, Test(OOD)=35


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICEM, SMILESsnamenM_ICMA, SMILESsnamenM_ICMM, SMILESsnamenM_MOPFP, SMILESsnamenM_NCMA, SMILESsnamenM_NCMM, SMILESsnamenM_PCBM, SMILESsnamenM_PFP, SMILESsnamenM_TC61BM, SMILESsnamenM_TC61PM, SMILESsnamenM_TFP, SMILESsnamep1M_BDTH1, SMILESsnamep1M_BDTH11, SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH6, SMILESsnamep1M_BDTH7, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_DTSCh, SMILESsnamep1M_DTTPTP1, SMILESsnamep1M_DTTPTP5, SMILESsnamep1M_PBDTTEHDTHBTffP1, SMILESsnamep1M_PBDTTPD, SMILESsnamep1M_PCDTBX, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PCDTPT, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PCDTQx, SMILESsnamep1M_PCDTTPP, SMILESsnamep1M_PDTPDTDPP, SMILESsnamep1M_PFDTBT, SMILESsnamep1M_PICDTBT, SMILESsnamep1M_PTB1, SMILESsnamep1M_PTB2, SMILESsnamep1M_PTB6, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2.thienyl..1.2.5.oxadiazo

...


--- Training PLS for: PCEmax  ---
  OOD Split: Total=1045, Train=941, Test(OOD)=104


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.


<U+2705> Fixed PLS Summary saved as: fixed20250625_pls_summary_20251212.csv 
